In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import os

In [ ]:
base_dir = os.getcwd()

In [ ]:
img_dir = os.path.join(base_dir, 'images')


In [19]:
# Define the data transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to the same size as expected by ResNet
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize with ImageNet stats
])

dataset = datasets.ImageFolder(img_dir, transform=transform)

dataset.samples.sort(key=lambda x: x[0])
dataset.imgs = dataset.samples

g = torch.Generator()
g.manual_seed(42)

train_size = int(0.8 * len(dataset))
test_size = int(0.1 * len(dataset))
val_size = len(dataset) - train_size - test_size

train_dataset, test_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, test_size, val_size])

batch_size = 64

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

print(f"Train dataset size: {len(train_loader.dataset)}")
print(f"Test dataset size: {len(test_loader.dataset)}")
print(f"Validation dataset size: {len(val_loader.dataset)}")

Train dataset size: 8000
Test dataset size: 1000
Validation dataset size: 1000


In [12]:
dataset.class_to_idx

{'closed': 0, 'open': 1}

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the pre-trained ResNet-18 model
model = models.resnet18(pretrained=True)

num_classes = 2
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

/opt/homebrew/Caskroom/miniconda/base/envs/pytorch/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/pytorch/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/koyun/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 59.4MB/s]


In [15]:
# Train model
num_epochs = 20
for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}")



Epoch 1/20, Loss: 0.201014137968421
Epoch 2/20, Loss: 0.03309361470863223
Epoch 3/20, Loss: 0.01608358507975936
Epoch 4/20, Loss: 0.00876132880244404
Epoch 5/20, Loss: 0.0051177781485021116
Epoch 6/20, Loss: 0.004335538221988827
Epoch 7/20, Loss: 0.0033102030172012745
Epoch 8/20, Loss: 0.001965892498847097
Epoch 9/20, Loss: 0.0024084763166029007
Epoch 10/20, Loss: 0.0018666162912268191
Epoch 11/20, Loss: 0.0017973940938245505
Epoch 12/20, Loss: 0.002466717927251011
Epoch 13/20, Loss: 0.0013300974356825464
Epoch 14/20, Loss: 0.0011176618796307593
Epoch 15/20, Loss: 0.0012119229952804744
Epoch 16/20, Loss: 0.0008807655993150548
Epoch 17/20, Loss: 0.0007512074747937731
Epoch 18/20, Loss: 0.0006846677547437138
Epoch 19/20, Loss: 0.0009304290212457999
Epoch 20/20, Loss: 0.0008080953173339367


In [17]:
# Save the trained model
torch.save(model.state_dict(), 'resnet18-e_20-d_10k.pth')

In [21]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load the trained model
model = models.resnet18(pretrained=False)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model.load_state_dict(torch.load('resnet18-e_20-d_10k.pth'))
model = model.to(device)
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)  
        labels = labels.to(device)

        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

accuracy = accuracy_score(y_true, y_pred)
confusion_mat = confusion_matrix(y_true, y_pred)
classification_rep = classification_report(y_true, y_pred)

print(f"Validation Accuracy: {accuracy}")
print("Classification Report:")
print(classification_rep)

/opt/homebrew/Caskroom/miniconda/base/envs/pytorch/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/pytorch/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/var/folders/wm/slshhxgx0z9_dzk3hdzqng0m0000gn/T/ipykernel_2342/2410172799.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for

Validation Accuracy: 1.0
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       494
           1       1.00      1.00      1.00       506

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

